# NB2 — Does path structure still matter once variance is fixed?

**Question.** After calibrating the forward scale and checking the backward scale, how much does
the skip weight α change how well a deep residual MLP trains under a fixed budget, and does it act
only through the effective depth L(1−α²)?

**Contents.** 0 Hypotheses · 1 Design · 2 Core code · 3 Control checks · 4 Learning-rate sweep ·
5 Final runs · 6 Results · 7 Mechanism · 8 Effective-depth test · 9 Sensitivity check ·
10 Optional parameter-matched control · 11 Conclusions and scope

In [1]:
QUICK = True  # Submission target: cached runs in under five minutes on the lab kernel.

## Setup

In [2]:
import time
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

import config as C
from utils import (env_report, get_device, set_seed, make_generator, load_fashion_mnist,
                   iterate_minibatches, evaluate, save_result, result_exists, load_results,
                   summarise, plot_mean_sd, plot_seed_points)

T0 = time.time()
ENV = env_report()
DEVICE = get_device()
DATA = load_fashion_mnist(device=DEVICE, n_val=C.N_VAL, split_seed=C.SPLIT_SEED,
                          n_calib=C.N_CALIB, calib_seed=C.CALIB_SEED)
print({k: tuple(v.shape) for k, v in DATA.items() if torch.is_tensor(v)})

         python: 3.9.6
       platform: macOS-26.6.2-arm64-arm-64bit
          torch: 2.8.0
          numpy: 2.0.2
 cuda_available: False
  mps_available: True
         device: mps
    torchvision: 0.23.0
     matplotlib: 3.9.4
{'x_train': (50000, 784), 'y_train': (50000,), 'x_val': (10000, 784), 'y_val': (10000,), 'x_test': (10000, 784), 'y_test': (10000,), 'x_calib': (512, 784)}


## 0. Hypotheses and pre-registration

Commit:

Date:

Hypotheses and decision rules:

## 1. Design

_TODO (argued, not just stated):_
- Why α is the manipulated variable, and why the 16-block runs are a repetition of the same
  manipulation at a second depth rather than a second manipulated variable.
- Why the learning rate is the nuisance tuned per condition: the relative step size of SGD depends
  on the weight scale, which changes with α, so one shared learning rate would favour some α.
- Why momentum, batch size and epochs are fixed rather than tuned (standard values, weak
  interaction with the initialisation compared with the learning rate, compute budget), and that
  the sensitivity check tests the epoch choice.
- Why these α values, the exact matched-pair partners, and more seeds for the matched-pair conditions.

In [3]:
print(f"{'depth':>5} {'alpha':>7} {'L(1-a^2)':>9}")
for depth, alphas in C.ALPHAS.items():
    for a in alphas:
        print(f"{depth:>5} {a:>7.4f} {C.effective_depth(depth, a):>9.2f}")
print("matched pairs:", [(p, q, round(C.effective_depth(*p), 2), round(C.effective_depth(*q), 2))
                         for p, q in C.MATCHED_PAIRS])

depth   alpha  L(1-a^2)
   32  0.0000     32.00
   32  0.5000     24.00
   32  0.7071     16.00
   32  0.7906     12.00
   32  0.9000      6.08
   32  0.9700      1.89
   16  0.0000     16.00
   16  0.5000     12.00
   16  0.7071      8.00
   16  0.7874      6.08
   16  0.9000      3.04
   16  0.9390      1.89
   16  0.9700      0.95
matched pairs: [((32, 0.7071067811865476), (16, 0.0), 16.0, 16.0), ((32, 0.7905694150420949), (16, 0.5), 12.0, 12.0), ((32, 0.9), (16, 0.7874007874011811), 6.08, 6.08), ((32, 0.97), (16, 0.9390420650854784), 1.89, 1.89)]


## 2. Core code

Same definitions as NB1 §1, §4 and §5 so that this notebook runs on its own.
**Keep them identical to NB1**: if you change one, change both.

In [4]:
class ResMLP(nn.Module):
    """Pre-activation residual MLP.

    input  : Linear(784 -> D, bias=False)                  -> h_0
    block l: h_{l+1} = alpha * h_l + W_l(ReLU(h_l))         W_l = Linear(D -> D, bias=False)
    output : Linear(D -> 10, bias=False) applied to h_L      (logits)

    forward(x, return_hidden=False) -> logits, or (logits, [h_0, ..., h_L]) if return_hidden.
    """

    def __init__(self, depth: int, width: int = C.WIDTH, alpha: float = 0.0):
        super().__init__()
        self.alpha = alpha
        self.input_layer = nn.Linear(C.IN_DIM, width, bias=False)
        self.blocks = nn.ModuleList(
            nn.Linear(width, width, bias=False) for _ in range(depth)
        )
        self.output_layer = nn.Linear(width, C.N_CLASSES, bias=False)

    def forward(self, x: torch.Tensor, return_hidden: bool = False):
        x = x.flatten(start_dim=1)
        h = self.input_layer(x)

        if return_hidden:
            hidden = [h]

        for block in self.blocks:
            h = self.alpha * h + block(torch.relu(h))
            if return_hidden:
                hidden.append(h)

        logits = self.output_layer(h)
        if return_hidden:
            return logits, hidden
        return logits

In [5]:
@torch.no_grad()
def init_weights(model: ResMLP, alpha: float, sigma2_block: "float | None" = None) -> None:
    width = model.input_layer.out_features

    if sigma2_block is None:
        sigma2_block = 2 * (1 - alpha ** 2) / width

    model.input_layer.weight.normal_(
        mean=0.0,
        std=model.input_layer.in_features ** -0.5,
    )
    for block in model.blocks:
        block.weight.normal_(mean=0.0, std=sigma2_block ** 0.5)

    model.output_layer.weight.normal_(mean=0.0, std=width ** -0.5)

In [6]:
@torch.no_grad()
def calibrate(model: ResMLP, x_calib: torch.Tensor, alpha: float) -> list[float]:
    """Rescale each block's W so that E[h_{l+1}^2] = E[h_l^2] exactly on x_calib.

    For block l with f = W_l(ReLU(h_l)):  a = E[f^2], b = E[h_l * f], c = E[h_l^2]
        s = (-alpha * b + sqrt(alpha^2 * b^2 + (1 - alpha^2) * a * c)) / a,   W_l <- s * W_l
    Process the blocks in order, so block l+1 sees the already-calibrated h_{l+1}.
    Returns the list of scale factors s (they should all be close to 1).
    """
    h = model.input_layer(x_calib.flatten(start_dim=1))
    scales = []

    for block in model.blocks:
        branch = block(torch.relu(h))
        a = branch.square().mean()
        b = (h * branch).mean()
        c = h.square().mean()

        if a.item() == 0:
            raise ValueError("Cannot calibrate a residual branch with zero output.")

        root = torch.sqrt((alpha * b).square() + (1 - alpha**2) * a * c)
        scale = (root - alpha * b) / a

        block.weight.mul_(scale)
        h = alpha * h + scale * branch
        scales.append(scale.item())

    return scales

In [7]:
def train_one_run(depth: int, alpha: float, lr: float, seed: int, epochs: int = C.EPOCHS,
                  width: int = C.WIDTH, data: dict = None, save: bool = True,
                  with_test: bool = False, n_train: "int | None" = None, tag: str = "") -> dict:
    """Train one network; return (and by default save) its result record.

    set_seed(seed) -> ResMLP(depth, width, alpha) -> init_weights -> calibrate on data["x_calib"]
    -> torch.optim.SGD(lr=lr, momentum=C.MOMENTUM), batches of C.BATCH_SIZE shuffled with
       make_generator(seed)
    -> after every epoch: evaluate() on the full training and validation sets
    -> if the loss becomes NaN/inf, stop and set diverged = True (keep the record).

    Record keys: depth, width, alpha, lr, seed, epochs, train_loss, val_loss, train_acc, val_acc
    (lists per epoch; train_loss[C.EARLY_EPOCH - 1] is the early-training metric),
    final_train_loss, diverged, seconds, and test_loss / test_acc when
    with_test (final runs only). n_train trains on a subset (QUICK live example; then save=False).
    """
    data = DATA if data is None else data
    set_seed(seed)
    device = data["x_calib"].device
    model = ResMLP(depth=depth, width=width, alpha=alpha).to(device)

    init_weights(model, alpha)
    calibrate(model, data["x_calib"], alpha)

    optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=C.MOMENTUM)
    generator = make_generator(seed)
    x_train = data["x_train"]
    y_train = data["y_train"]
    if n_train is not None:
        x_train = x_train[:n_train]
        y_train = y_train[:n_train]
        save = False 

    train_loss, val_loss = [], []
    train_acc, val_acc = [], []
    diverged = False
    start_time = time.perf_counter()

    for epoch in range(epochs):
        model.train()
        for xb, yb in iterate_minibatches(
            x_train, y_train, C.BATCH_SIZE, generator=generator
        ):
            optimizer.zero_grad()
            logits = model(xb)
            loss = nn.functional.cross_entropy(logits, yb)

            if not torch.isfinite(loss).item():
                diverged = True
                break

            loss.backward()
            optimizer.step()

        if diverged:
            break

        train_metrics = evaluate(model, x_train, y_train)
        val_metrics = evaluate(model, data["x_val"], data["y_val"])

        if not (np.isfinite(train_metrics["loss"]) and np.isfinite(val_metrics["loss"])):
            diverged = True
            break

        train_loss.append(train_metrics["loss"])
        train_acc.append(train_metrics["acc"])
        val_loss.append(val_metrics["loss"])
        val_acc.append(val_metrics["acc"])
        
    record = {
        "depth": depth,
        "width": width,
        "alpha": alpha,
        "lr": lr,
        "seed": seed,
        "epochs": epochs,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "train_acc": train_acc,
        "val_acc": val_acc,
        "final_train_loss": None if diverged else train_loss[-1],
        "diverged": diverged,
        "seconds": time.perf_counter() - start_time,
    }

    if with_test:
        record["test_loss"] = None
        record["test_acc"] = None
        if not diverged:
            test_metrics = evaluate(model, data["x_test"], data["y_test"])
            record["test_loss"] = test_metrics["loss"]
            record["test_acc"] = test_metrics["acc"]

    if save:
        save_result(record, tag=tag)

    return record

In [8]:
trial = train_one_run(
    depth=2,
    alpha=C.ALPHA_BASELINE,
    lr=0.001,
    seed=0,
    epochs=1,
    n_train=256,
    save=False,
)

print(trial["diverged"], len(trial["train_loss"]), trial["final_train_loss"])

False 1 2.491852283477783


## 3. Control check: forward and backward second moments at initialisation

Measurements:

In [ ]:
# TODO

## 4. Learning-rate sweep (seed 0)

Each condition is swept over `C.LR_GRID`; the grid is extended where the best value or an edge
of the usable range sits on the grid edge. Runs already in `results/` are skipped, so the sweep
can be resumed.

In [ ]:
def sweep(depth: int, alpha: float, lr_grid=C.LR_GRID) -> list[dict]:
    """Run the seed-0 learning-rate sweep for one condition."""

    records = []
    # Remove duplicates and sort learning rates.
    lr_grid = sorted({_lr_key(lr) for lr in lr_grid})

    for lr in lr_grid:
        if lr <= 0 or not np.isfinite(lr):
            raise ValueError(f"Invalid learning rate: {lr}")

        if result_exists(
            depth=depth,
            alpha=alpha,
            lr=lr,
            seed=C.TUNE_SEED,
            epochs=C.EPOCHS,
            tag="",
            width=C.WIDTH,
        ):
            print(f"Skip existing run: lr={lr:g}", flush=True)

            record = _load_one(
                depth, alpha, lr, C.TUNE_SEED, C.WIDTH, ""
            )
        else:
            if QUICK:
                raise RuntimeError(
                    f"QUICK mode: missing sweep result for "
                    f"depth={depth}, alpha={alpha}, lr={lr:g}"
                )

            print(f"Training: lr={lr:g}", flush=True)

            record = train_one_run(
                depth=depth,
                alpha=alpha,
                lr=lr,
                seed=C.TUNE_SEED,
                epochs=C.EPOCHS,
                width=C.WIDTH,
                save=True,
                tag="",
            )

        records.append(record)

    return records



# τ = 1.25 × L_base, fixed from the seed-0 baseline at the learning rate the selection rule
# picks for it (32 blocks, α = 1/√2). A learning rate is usable when the seed-0 run does
# not diverge and its final training loss is at most τ. hypotheses.md § "Usable
# learning-rate threshold" and § "Learning-rate grid, extension and tie-break".
TIEBREAK_SEED = 6          # outside the final seeds 1–5; same extra seed as the baseline
TIEBREAK_TAG = "tiebreak"
TIE_REL = 0.05
LR_EXTEND = 3.0
MAX_LR_EXTENSIONS = 8


def _lr_key(lr: float) -> float:
    return float(f"{float(lr):.8g}")


def _final_loss(record: dict):
    if record.get("diverged") or record.get("final_train_loss") is None:
        return None
    loss = float(record["final_train_loss"])
    if not np.isfinite(loss):
        return None
    return loss


def _within_rel(a: float, b: float, rel: float = TIE_REL) -> bool:
    """True when the two losses differ by strictly less than `rel` of the lower one."""
    lo = min(a, b)
    if lo <= 0:
        return a == b
    return (max(a, b) - lo) / lo < rel


def usable_tau() -> float:
    """tau = 1.25 x the seed-0 baseline loss at the learning rate the selection rule picks.

    Reads the untagged seed-0 sweep of the baseline condition at the main epoch budget and takes
    the lowest final loss, which is the same rule select_lr() applies. Deriving tau here rather
    than hard-coding a learning rate keeps it correct when the budget changes.
    """
    runs = load_results(
        tag="",
        depth=C.DEPTH_MAIN,
        width=C.WIDTH,
        alpha=C.ALPHA_BASELINE,
        seed=C.TUNE_SEED,
        epochs=C.EPOCHS,
    )
    finished = [record for record in runs if _final_loss(record) is not None]
    if not finished:
        raise RuntimeError(
            f"no finished seed-0 baseline sweep at {C.EPOCHS} epochs; run sweep() first"
        )
    best = min(finished, key=_final_loss)
    return 1.25 * _final_loss(best)


def _is_usable(record: dict, tau: float) -> bool:
    loss = _final_loss(record)
    return loss is not None and loss <= tau


def _load_one(depth, alpha, lr, seed, width, tag):
    found = load_results(
        depth=depth, alpha=alpha, lr=_lr_key(lr), seed=seed,
        epochs=C.EPOCHS, width=width, tag=tag,
    )
    if len(found) != 1:
        raise RuntimeError(
            f"expected one saved run depth={depth} alpha={alpha} lr={_lr_key(lr):g} "
            f"seed={seed} tag={tag!r}; found {len(found)}"
        )
    return found[0]


def _ensure_run(depth, alpha, lr, seed, width, tag):
    """Return the saved run, training it only when this is not a QUICK session."""
    lr = _lr_key(lr)
    if result_exists(depth, alpha, lr, seed, C.EPOCHS, tag=tag, width=width):
        return _load_one(depth, alpha, lr, seed, width, tag)
    if QUICK:
        raise RuntimeError(
            "QUICK mode will not train a missing run: "
            f"depth={depth} alpha={alpha} lr={lr:g} seed={seed} tag={tag!r}"
        )
    print(f"depth={depth} alpha={alpha:.4f} lr={lr:g} seed={seed}: training", flush=True)
    return train_one_run(
        depth=depth, alpha=alpha, lr=lr, seed=seed, epochs=C.EPOCHS,
        width=width, save=True, tag=tag,
    )


def _seed0_by_lr(depth, alpha, width):
    found = load_results(
        depth=depth, alpha=alpha, width=width, seed=C.TUNE_SEED,
        epochs=C.EPOCHS, tag="",
    )
    by_lr = {}
    for record in found:
        key = _lr_key(record["lr"])
        if key in by_lr:
            raise RuntimeError(
                f"duplicate seed-0 run at depth={depth} alpha={alpha} lr={key:g}"
            )
        by_lr[key] = record
    return by_lr


def _extend_grid(by_lr, depth, alpha, width, tau):
    """Train ×3 / ÷3 points until both ends fail and the best seed-0 loss is interior.

    An end fails when that run diverges or its final loss is above τ. A side is still
    open when its endpoint is usable, or when the best loss sits on that endpoint.
    """
    steps = 0
    while True:
        grid = sorted(by_lr)
        finite = [lr for lr in grid if _final_loss(by_lr[lr]) is not None]
        if not finite:
            raise RuntimeError(
                f"depth={depth} alpha={alpha}: every seed-0 learning rate diverged"
            )
        best = min(finite, key=lambda lr: _final_loss(by_lr[lr]))
        lo, hi = grid[0], grid[-1]
        extend_lo = best == lo or _is_usable(by_lr[lo], tau)
        extend_hi = best == hi or _is_usable(by_lr[hi], tau)
        if not extend_lo and not extend_hi:
            return by_lr
        if steps >= MAX_LR_EXTENSIONS:
            raise RuntimeError(
                f"depth={depth} alpha={alpha:.4f}: learning-rate grid still open after "
                f"{MAX_LR_EXTENSIONS} extensions (ends {lo:g} and {hi:g}, best={best:g})"
            )
        steps += 1
        if extend_lo:
            new_lr = _lr_key(lo / LR_EXTEND)
            if new_lr >= lo:
                raise RuntimeError(f"refusing to extend the low end from {lo:g} to {new_lr:g}")
            print(f"depth={depth} alpha={alpha:.4f}: extend low end to lr={new_lr:g}", flush=True)
            by_lr[new_lr] = _ensure_run(depth, alpha, new_lr, C.TUNE_SEED, width, "")
        if extend_hi:
            new_lr = _lr_key(hi * LR_EXTEND)
            if new_lr <= hi:
                raise RuntimeError(f"refusing to extend the high end from {hi:g} to {new_lr:g}")
            print(f"depth={depth} alpha={alpha:.4f}: extend high end to lr={new_lr:g}", flush=True)
            by_lr[new_lr] = _ensure_run(depth, alpha, new_lr, C.TUNE_SEED, width, "")


def select_lr(depth: int, alpha: float, width: int = C.WIDTH) -> float:
    """Learning rate for one condition, from the seed-0 sweep.

    The seed-0 run with the lowest final training loss wins. If that learning rate,
    or either end of the usable range, sits on the edge of the grid, train outward
    by ×3 and ÷3 until both ends fail. If the winner and a neighbouring learning rate
    differ by less than 5%, train one extra seed and compare the means; if those
    means are still within 5%, take the smaller learning rate.
    """
    tau = usable_tau()
    by_lr = _seed0_by_lr(depth, alpha, width)
    missing = [lr for lr in C.LR_GRID if _lr_key(lr) not in by_lr]
    if missing:
        raise RuntimeError(
            f"depth={depth} alpha={alpha:.4f}: seed-0 sweep is missing "
            f"{[_lr_key(lr) for lr in missing]}. Run sweep() first."
        )
    _extend_grid(by_lr, depth, alpha, width, tau)

    grid = sorted(by_lr)
    finite = [lr for lr in grid if _final_loss(by_lr[lr]) is not None]
    if not finite:
        raise RuntimeError(
            f"depth={depth} alpha={alpha}: every seed-0 learning rate diverged"
        )
    best = min(finite, key=lambda lr: (_final_loss(by_lr[lr]), lr))
    best_loss = _final_loss(by_lr[best])
    i = grid.index(best)
    tied = [best]
    for j in (i - 1, i + 1):
        if not 0 <= j < len(grid):
            continue
        neighbour = _final_loss(by_lr[grid[j]])
        if neighbour is not None and _within_rel(neighbour, best_loss):
            tied.append(grid[j])

    if len(tied) == 1:
        chosen = best
        note = ""
    else:
        means = {}
        for lr in tied:
            extra = _ensure_run(depth, alpha, lr, TIEBREAK_SEED, width, TIEBREAK_TAG)
            extra_loss = _final_loss(extra)
            means[lr] = (
                float("inf") if extra_loss is None else (best_loss if lr == best else _final_loss(by_lr[lr]) + extra_loss) / 2
                if extra_loss is not None else float("inf")
            )
            if extra_loss is None:
                means[lr] = float("inf")
            else:
                means[lr] = (_final_loss(by_lr[lr]) + extra_loss) / 2.0
        finite_means = {lr: m for lr, m in means.items() if np.isfinite(m)}
        if not finite_means:
            raise RuntimeError(
                f"depth={depth} alpha={alpha}: tie-break seed {TIEBREAK_SEED} "
                "diverged for every close learning rate"
            )
        best_mean = min(finite_means.values())
        close = [lr for lr, m in finite_means.items() if _within_rel(m, best_mean)]
        chosen = min(close)
        if len(close) > 1:
            note = (
                f"; tie-break seed {TIEBREAK_SEED}, means still within 5%, "
                "took the smaller lr"
            )
        else:
            note = f"; tie-break seed {TIEBREAK_SEED}, took the lower mean"

    print(
        f"depth={depth} alpha={alpha:.4f}: selected lr={chosen:g} (τ={tau:.4f}{note})",
        flush=True,
    )
    return chosen


if not QUICK:
    for depth, alphas in C.ALPHAS.items():
        for alpha in alphas:
            sweep(depth, alpha)

## 5. Final runs (seeds 1–3; 1–5 for the matched-pair conditions)

In [ ]:
if not QUICK:
    pass  # TODO: for every condition and every s in C.final_seeds(depth, alpha):
          #       train_one_run(..., lr=select_lr(...), seed=s, with_test=True)

## 6. Results

- 6a. Training-loss curves per α (32 blocks), mean ± sd over seeds.
- 6b. Final training loss against α (32 blocks), one point per seed.
- 6c. Final training loss against learning rate for every condition: shows tuning is fair and gives
  the usable range.
- 6d. Training loss after epoch `C.EARLY_EPOCH` against α: training speed, useful if the final
  losses are close.

_TODO: apply the decision rules from §0 and say what each figure shows. Diverged runs are listed,
not dropped._

In [ ]:
# TODO: figures 6a-6d

## 7. Mechanism: the input–output Jacobian at initialisation

For each condition, J = ∂h_L/∂h_0 at 32 inputs: similarity of J at nearby inputs (gradient
shattering, UDL §11.1), stable rank ‖J‖_F² / ‖J‖₂², and the spectrum of squared singular values
(dynamical isometry, Pennington et al. 2017).

In [ ]:
def jacobian_metrics(model: ResMLP, h0: torch.Tensor, eps_scale: float = 0.1,
                     generator: torch.Generator | None = None) -> dict:
    """h0: (n, D) inputs to the first block. For each row, J = d h_L / d h_0 (D x D) via
    torch.autograd.functional.jacobian on the block stack only.
    Returns the mean cosine similarity between J(h0) and J(h0 + eps), eps = eps_scale * std(h0)
    * N(0, I); the mean stable rank ||J||_F^2 / sigma_max^2 (torch.linalg.svdvals); and the
    singular values for the spectrum plot."""
    # TODO
    raise NotImplementedError

# TODO: compute for every condition; plot against effective depth

## 8. Effective-depth prediction

Compare final training loss at the selected learning rate. Use the four exact pairs in config; for two other points interpolate the 32-block means in log effective depth. Do not extrapolate to (16, 0.97). Error = observed 16-block mean minus prediction.

For the synthetic stage-0 demonstration, see the README.

Statistical decision rules:

Experimental results:

In [ ]:
def effective_depth_test(records: list[dict], *, selected_lrs: dict) -> list[dict]:
    """Stage 0: point estimates only. selected_lrs maps (depth, exact alpha) to lr.

    Supply main-run records and the externally selected learning rates. Check final
    seeds before averaging; do not silently average an incomplete condition.
    """
    import math
    from bisect import bisect_left
    from statistics import mean

    groups = {(L, a): {} for L, alphas in C.ALPHAS.items() for a in alphas}
    for r in records:
        key = (r["depth"], r["alpha"])
        if (key not in groups or r["width"] != C.WIDTH or r["epochs"] != C.EPOCHS
                or r.get("_meta", {}).get("tag", "") or r["seed"] not in C.final_seeds(*key)
                or r["lr"] != selected_lrs[key]):
            continue
        seed = r["seed"]
        if seed in groups[key]:
            raise ValueError(f"Duplicate final: {key}, seed={seed}")
        if r["diverged"] or not math.isfinite(float(r["final_train_loss"])):
            raise ValueError(f"Diverged or nonfinite final: {key}, seed={seed}")
        groups[key][seed] = float(r["final_train_loss"])

    means = {}
    for key, losses in groups.items():
        missing = set(C.final_seeds(*key)) - losses.keys()
        if missing:
            raise ValueError(f"Missing finals: {key}, seeds={sorted(missing)}")
        means[key] = mean(losses.values())

    partners = {q: p for p, q in C.MATCHED_PAIRS}
    curve = sorted((C.effective_depth(32, a), means[32, a]) for a in C.ALPHAS[32])
    depths = [d for d, _ in curve]
    rows = []
    for a in C.ALPHAS[16]:
        key = (16, a)
        d = C.effective_depth(*key)
        if key in partners:  # use configured exact pairs, not rounded depth labels
            predicted, method = means[partners[key]], "matched"
        elif depths[0] < d < depths[-1]:
            i = bisect_left(depths, d)
            (lo, left), (hi, right) = curve[i-1:i+1]
            weight = math.log(d / lo) / math.log(hi / lo)
            predicted, method = left + weight * (right - left), "interpolated"
        else:
            continue  # no extrapolation
        rows.append(dict(alpha=a, effective_depth=d, method=method, observed=means[key],
                         predicted=predicted, error=means[key] - predicted))
    return rows

## 9. Sensitivity check: 32 blocks, 12 epochs, selected learning rates

In [ ]:
if not QUICK:
    pass  # TODO: train_one_run(..., epochs=C.EPOCHS_SENSITIVITY, tag="sens") for the final seeds

# TODO: does the shape of loss against alpha survive longer training?

## 10. Optional parameter-matched control

Design: see `C.PARAM_MATCHED`.

Results:

In [ ]:
if not QUICK:
    pass  # TODO (only if time allows, priority 4)

## 11. Conclusions and scope

In [ ]:
print(f"Total runtime: {time.time() - T0:.0f} s  (QUICK = {QUICK}; must stay under 300 s when QUICK)")